# NN-kNN Classification Workflow

This notebook uses the maintained IJCAI-26 NN-kNN core for classification. Retrieval, feature weighting, case scoring, and case normalization remain the current model; the output layer sums normalized case activation into class probability mass and trains it with negative log likelihood.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.classification_workflow import (
    list_supported_classification_datasets,
    list_supported_classification_benchmark_methods,
    make_classification_cfg,
    run_single_nnknn_classification_experiment,
    run_repeated_classification_model_benchmarks,
)

print(list_supported_classification_datasets())
print(list_supported_classification_benchmark_methods())

## Small Dataset Sanity Check

The IJCAI-25 small-data family is available as `iris`, `zebra`, `zebra_special`, `wine`, `breast_cancer`, `balance`, and `digits`. Standardization is fitted on the training split only.

In [ ]:
cfg = make_classification_cfg({
    "training_epochs": 10,
    "batch_size": 32,
    "patience": 5,
    "tau": 0.5,  # Representative small-data setting; retune per reporting protocol.
    "case_normalizer": "softmax",  # Change to "sparsemax" for sparse case activation.
    "top_k": 5,
    "explanation_mode": True,
    "checkpoint_path": "checkpoints/nnknn_classification_notebook.pth",
})

result = run_single_nnknn_classification_experiment(
    "iris", cfg, run_seed=42, split_seed=42, checkpoint_label="iris_demo"
)
print("Validation accuracy:", result["accuracy"])
print("First class probability masses:", result["class_probabilities"][:3])

In [ ]:
# Explanation output uses cases retrieved by the current model.
print("Top retrieved class ids for query 0:", result["most_activated_class_ids"][0])
print("Top retrieved activation mass for query 0:", result["most_activated_activations"][0])

## Representative Benchmark Check

This short run checks that NN-kNN, kNN, and a four-hidden-layer MLP execute on shared stratified folds. Use more folds and epochs for a table intended for reporting.

In [ ]:
run_tabular_benchmark = False
if run_tabular_benchmark:
    summary, runs, _ = run_repeated_classification_model_benchmarks(
        "iris",
        cfg,
        methods=["nnknn", "knn", "mlp"],
        num_runs=3,
        mode="kfold",
        base_seed=42,
        method_cfgs={"mlp": {"epochs": 50, "patience": 10}},
    )
    display(summary)

## Image Workflow

For `mnist`, `cifar10`, or `svhn`, the loader uses the official train/test split and fits normalization statistics on training images. Training reserves an inner validation slice for checkpoint selection and reports on official test data. Start with a subset before a full image benchmark.

In [ ]:
run_image_demo = False
if run_image_demo:
    image_cfg = make_classification_cfg({
        "training_epochs": 5,
        "batch_size": 64,
        "top_k": 5,
        "explanation_mode": True,
        "checkpoint_path": "checkpoints/nnknn_mnist_subset.pth",
    })
    image_result = run_single_nnknn_classification_experiment(
        "mnist",
        image_cfg,
        dataset_kwargs={"max_train_samples": 1000, "max_eval_samples": 300},
        checkpoint_label="mnist_subset",
    )
    print("MNIST subset accuracy:", image_result["accuracy"])

For image baseline comparisons, call `run_repeated_classification_model_benchmarks` with methods `convnet`, `knn_pixels`, `knn_conv_frozen`, `nnknn_conv_trainable`, and `nnknn_conv_frozen`. The frozen NN-kNN method reuses a trained ConvNet feature extractor and keeps it frozen during NN-kNN training.